In [1]:
# ============================================================
# WIN PROBABILITY TRAINING / VALIDATION NOTEBOOK
# Uses saved PA outcome model to simulate historical games
# and evaluate projected win probabilities vs real outcomes
# ============================================================

import os
import json
import math
import random
import warnings
from functools import lru_cache

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import log_loss, brier_score_loss

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

# Paths
MODEL_PATH = "model_pa.cbm"
MODEL_META_PATH = "model_meta.json"
MODEL_DF_PATH = "model_df.parquet"

print("setup complete")
print("working directory:", os.getcwd())
print("model exists:", os.path.exists(MODEL_PATH))
print("meta exists:", os.path.exists(MODEL_META_PATH))
print("df exists:", os.path.exists(MODEL_DF_PATH))

setup complete
working directory: c:\Users\Matthew\OneDrive\Desktop\machine learning- baseball
model exists: True
meta exists: True
df exists: True


In [2]:
# ----------------------------
# Load saved PA model + metadata
# ----------------------------

pa_model = CatBoostClassifier()
pa_model.load_model(MODEL_PATH)

with open(MODEL_META_PATH, "r") as f:
    model_meta = json.load(f)

model_df = pd.read_parquet(MODEL_DF_PATH)

feature_cols = model_meta.get("feature_cols", [])
cat_cols = model_meta.get("cat_cols", [])
target_col = model_meta.get("target_col", "pa_outcome")

print("PA model loaded.")
print("target_col:", target_col)
print("num feature cols:", len(feature_cols))
print("num categorical cols:", len(cat_cols))
print("model_df shape:", model_df.shape)

if len(feature_cols) > 0:
    print("\nFirst 15 features:")
    print(feature_cols[:15])

PA model loaded.
target_col: pa_outcome
num feature cols: 0
num categorical cols: 6
model_df shape: (3741396, 20)


In [3]:
# ----------------------------
# Quick inspection of available columns
# ----------------------------

print("Columns in model_df:")
print(model_df.columns.tolist())

print("\nSample rows:")
display(model_df.head(3))

Columns in model_df:
['pitcher', 'batter', 'pitch_type', 'balls', 'strikes', 'outs_when_up', 'inning', 'stand', 'p_throws', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z', 'zone', 'runner_1b', 'runner_2b', 'runner_3b', 'run_diff', 'pa_outcome', 'ball_type']

Sample rows:


,pitcher,batter,pitch_type,balls,strikes,outs_when_up,inning,stand,p_throws,release_speed,release_spin_rate,pfx_x,pfx_z,zone,runner_1b,runner_2b,runner_3b,run_diff,pa_outcome,ball_type
0,657277,641658,SI,0,0,0,1,R,R,92.3,<NA>,-1.45,0.01,13,0,0,0,0,walk,not_in_play
1,657277,641658,SI,1,0,0,1,R,R,92.6,<NA>,-1.38,0.15,4,0,0,0,0,walk,not_in_play
2,657277,641658,SI,1,1,0,1,R,R,93.4,<NA>,-1.43,0.18,5,0,0,0,0,walk,not_in_play


In [4]:
# ----------------------------
# Check for game simulation columns
# ----------------------------

required_cols = [
    "game_pk",
    "inning",
    "inning_topbot",
    "outs_when_up",
    "batter",
    "pitcher",
    "stand",
    "p_throws",
    "on_1b",
    "on_2b",
    "on_3b",
    "home_team",
    "away_team",
    "bat_score",
    "fld_score"
]

available_required = [c for c in required_cols if c in model_df.columns]
missing_required = [c for c in required_cols if c not in model_df.columns]

print("available required columns:", available_required)
print("\nmissing required columns:", missing_required)

available required columns: ['inning', 'outs_when_up', 'batter', 'pitcher', 'stand', 'p_throws']

missing required columns: ['game_pk', 'inning_topbot', 'on_1b', 'on_2b', 'on_3b', 'home_team', 'away_team', 'bat_score', 'fld_score']


In [5]:
# ----------------------------
# Find candidate raw data files
# ----------------------------

from pathlib import Path

root = Path(".")
parquet_files = sorted(root.rglob("*.parquet"))

print("Found parquet files:\n")
for p in parquet_files:
    size_mb = p.stat().st_size / (1024 * 1024)
    print(f"{p} | {size_mb:.1f} MB")

Found parquet files:

data\statcast_2021_2025.parquet | 454.8 MB
data\statcast_2026_ytd.parquet | 0.0 MB
model_df.parquet | 33.1 MB
models\model_df.parquet | 91.1 MB


In [6]:
# ----------------------------
# Load raw Statcast parquet
# ----------------------------

RAW_PARQUET_PATH = r"data/statcast_2021_2025.parquet"  # change if needed

raw = pd.read_parquet(RAW_PARQUET_PATH)

print("raw shape:", raw.shape)
print("num columns:", len(raw.columns))

print("\nFirst 60 columns:")
print(raw.columns.tolist()[:60])

display(raw.head(3))

raw shape: (3846144, 119)
num columns: 119

First 60 columns:
['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3']


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches,season
0,FF,2021-11-02,93.7,1.39,6.72,"Smith, Will",493329,519293,field_out,hit_into_play,<NA>,<NA>,<NA>,<NA>,5,"Yuli Gurriel grounds out, shortstop Dansby Swa...",W,R,L,HOU,ATL,X,6,ground_ball,0,2,2021,0.57,1.21,0.190973,2.68051,<NA>,<NA>,488726,2,9,Bot,98.12,136.43,<NA>,<NA>,<NA>,<NA>,-4.348927,-136.263269,-7.335202,8.101619,29.384843,-15.743652,3.37,1.53,5,93.8,-28,93.4,2112,6.1,660906,518595,518692,645277,663586,621020,592696,628338,594807,54.39,0.074,0.068,0.0,1,0,0,2,69,3,4-Seam Fastball,0,7,0,7,7,0,0,7,Standard,Standard,146,-0.001,-0.164,<NA>,<NA>,0.077,0.164,93.8,-7,-7,0.001,0.001,31,37,32,37,1,3,3,2,<NA>,<NA>,1.39,0.57,-0.57,46.9,<NA>,<NA>,<NA>,<NA>,<NA>,2021
1,FF,2021-11-02,92.9,1.38,6.72,"Smith, Will",493329,519293,None,foul,<NA>,<NA>,<NA>,<NA>,5,Foul,W,R,L,HOU,ATL,S,<NA>,None,0,1,2021,0.9,1.34,0.01112,2.117733,<NA>,<NA>,488726,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-5.455433,-134.989926,-8.881689,12.188852,30.69001,-14.009571,3.37,1.53,156,75.0,15,92.6,2206,6.3,660906,518595,518692,645277,663586,621020,592696,628338,594807,54.22,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,69,2,4-Seam Fastball,0,7,0,7,7,0,0,7,Standard,Standard,145,0.0,-0.056,<NA>,<NA>,<NA>,0.056,88.0,-7,-7,0.001,0.001,31,37,32,37,1,3,3,2,<NA>,<NA>,1.32,0.9,-0.9,48.0,<NA>,<NA>,<NA>,<NA>,<NA>,2021
2,FF,2021-11-02,93.1,1.35,6.73,"Smith, Will",493329,519293,None,called_strike,<NA>,<NA>,<NA>,<NA>,6,Called Strike,W,R,L,HOU,ATL,S,<NA>,None,0,0,2021,0.81,1.52,0.782772,2.133925,<NA>,<NA>,488726,2,9,Bot,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,-3.230974,-135.201801,-9.255781,10.67848,31.699974,-11.7058,3.4,1.53,<NA>,<NA>,<NA>,92.5,2216,6.2,660906,518595,518692,645277,663586,621020,592696,628338,594807,54.3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,69,1,4-Seam Fastball,0,7,0,7,7,0,0,7,Standard,Standard,143,0.0,-0.049,<NA>,<NA>,<NA>,0.049,<NA>,-7,-7,0.001,0.001,31,37,32,37,1,3,3,2,<NA>,<NA>,1.14,0.81,-0.81,46.7,<NA>,<NA>,<NA>,<NA>,<NA>,2021


In [7]:
# ----------------------------
# Check raw parquet for historical game replay columns
# ----------------------------

needed_cols = [
    "game_pk",
    "game_date",
    "home_team",
    "away_team",
    "inning",
    "inning_topbot",
    "outs_when_up",
    "balls",
    "strikes",
    "batter",
    "pitcher",
    "stand",
    "p_throws",
    "on_1b",
    "on_2b",
    "on_3b",
    "bat_score",
    "fld_score",
    "events",
    "description"
]

present = [c for c in needed_cols if c in raw.columns]
missing = [c for c in needed_cols if c not in raw.columns]

print("present:", present)
print("\nmissing:", missing)

present: ['game_pk', 'game_date', 'home_team', 'away_team', 'inning', 'inning_topbot', 'outs_when_up', 'balls', 'strikes', 'batter', 'pitcher', 'stand', 'p_throws', 'on_1b', 'on_2b', 'on_3b', 'bat_score', 'fld_score', 'events', 'description']

missing: []


In [8]:
# ----------------------------
# Inspect columns related to runners / scores / game ids
# ----------------------------

keywords = ["game", "team", "inning", "score", "on_", "runner", "event", "description"]

matching_cols = []
for c in raw.columns:
    c_lower = c.lower()
    if any(k in c_lower for k in keywords):
        matching_cols.append(c)

print("Matching columns:\n")
print(matching_cols)

Matching columns:

['game_date', 'events', 'description', 'game_type', 'home_team', 'away_team', 'game_year', 'on_3b', 'on_2b', 'on_1b', 'inning', 'inning_topbot', 'game_pk', 'home_score', 'away_score', 'bat_score', 'fld_score', 'post_away_score', 'post_home_score', 'post_bat_score', 'post_fld_score', 'home_score_diff', 'bat_score_diff', 'n_priorpa_thisgame_player_at_bat', 'pitcher_days_since_prev_game', 'batter_days_since_prev_game', 'pitcher_days_until_next_game', 'batter_days_until_next_game']


In [9]:
# ----------------------------
# Build clean PA-level dataset from raw Statcast
# ----------------------------

PA_OUTCOME_MAP = {
    "field_out": "out",
    "force_out": "out",
    "grounded_into_double_play": "out",
    "double_play": "out",
    "fielders_choice_out": "out",
    "strikeout": "strikeout",
    "strikeout_double_play": "strikeout",
    "walk": "walk",
    "intent_walk": "walk",
    "single": "single",
    "double": "double",
    "triple": "triple",
    "home_run": "home_run"
}

keep_outcomes = set(PA_OUTCOME_MAP.keys())

pa_df = raw.copy()

# keep only rows that have an event we can map
pa_df = pa_df[pa_df["events"].isin(keep_outcomes)].copy()

# map to target classes
pa_df["pa_outcome"] = pa_df["events"].map(PA_OUTCOME_MAP)

print("pa_df shape after outcome filter:", pa_df.shape)
print("\nOutcome counts:")
print(pa_df["pa_outcome"].value_counts(dropna=False))

pa_df shape after outcome filter: (973370, 120)

Outcome counts:
pa_outcome
out          440136
strikeout    228458
single       141595
walk          84804
double        43720
home_run      30918
triple         3739
Name: count, dtype: int64


In [10]:
# ----------------------------
# Define model features
# ----------------------------

candidate_features = [
    "pitcher",
    "batter",
    "balls",
    "strikes",
    "outs_when_up",
    "inning",
    "inning_topbot",
    "stand",
    "p_throws",
    "on_1b",
    "on_2b",
    "on_3b",
    "bat_score",
    "fld_score",
    "home_team",
    "away_team",
    "release_speed",
    "release_spin_rate",
    "zone",
    "pfx_x",
    "pfx_z"
]

feature_cols = [c for c in candidate_features if c in pa_df.columns]
cat_cols = [c for c in [
    "pitcher",
    "batter",
    "inning_topbot",
    "stand",
    "p_throws",
    "home_team",
    "away_team"
] if c in feature_cols]

print("feature_cols:", feature_cols)
print("\ncat_cols:", cat_cols)

feature_cols: ['pitcher', 'batter', 'balls', 'strikes', 'outs_when_up', 'inning', 'inning_topbot', 'stand', 'p_throws', 'on_1b', 'on_2b', 'on_3b', 'bat_score', 'fld_score', 'home_team', 'away_team', 'release_speed', 'release_spin_rate', 'zone', 'pfx_x', 'pfx_z']

cat_cols: ['pitcher', 'batter', 'inning_topbot', 'stand', 'p_throws', 'home_team', 'away_team']


In [14]:
# ----------------------------
# Clean model dataset
# ----------------------------

extra_cols = ["pa_outcome", "game_date", "game_pk", "events"]
model_cols = list(dict.fromkeys(feature_cols + extra_cols))

train_df = pa_df[model_cols].copy()

# convert game_date
train_df["game_date"] = pd.to_datetime(train_df["game_date"], errors="coerce")
train_df = train_df.dropna(subset=["game_date", "pa_outcome"]).copy()

# numeric columns
numeric_cols = [c for c in feature_cols if c not in cat_cols]
for c in numeric_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce")

# fill missing numeric with median
for c in numeric_cols:
    med = train_df[c].median()
    train_df[c] = train_df[c].fillna(med)

# fill missing categorical with string token
for c in cat_cols:
    train_df[c] = train_df[c].astype("string").fillna("UNK").astype(str)

train_df["game_year"] = train_df["game_date"].dt.year

print("train_df shape:", train_df.shape)
print("\nYears available:")
print(train_df["game_year"].value_counts().sort_index())

print("\nDuplicate columns:")
print(train_df.columns[train_df.columns.duplicated()].tolist())

train_df shape: (973370, 26)

Years available:
game_year
2021    194135
2022    199275
2023    196010
2024    191815
2025    192135
Name: count, dtype: int64

Duplicate columns:
[]


In [15]:
# ----------------------------
# Train/test split by year
# ----------------------------

TRAIN_YEARS = [2021, 2022, 2023, 2024]
TEST_YEARS = [2025]

train_mask = train_df["game_year"].isin(TRAIN_YEARS)
test_mask = train_df["game_year"].isin(TEST_YEARS)

train_pa = train_df.loc[train_mask].copy()
test_pa = train_df.loc[test_mask].copy()

print("train_pa shape:", train_pa.shape)
print("test_pa shape:", test_pa.shape)

print("\nTrain years:")
print(train_pa["game_year"].value_counts().sort_index())

print("\nTest years:")
print(test_pa["game_year"].value_counts().sort_index())

train_pa shape: (781235, 26)
test_pa shape: (192135, 26)

Train years:
game_year
2021    194135
2022    199275
2023    196010
2024    191815
Name: count, dtype: int64

Test years:
game_year
2025    192135
Name: count, dtype: int64


In [17]:
# ----------------------------
# Train PA outcome model
# ----------------------------

from catboost import Pool

X_train = train_pa[feature_cols].copy()
y_train = train_pa["pa_outcome"].copy()

X_test = test_pa[feature_cols].copy()
y_test = test_pa["pa_outcome"].copy()

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("Duplicate X_train columns:", X_train.columns[X_train.columns.duplicated()].tolist())

train_pool = Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_cols,
    feature_names=feature_cols
)

test_pool = Pool(
    data=X_test,
    label=y_test,
    cat_features=cat_cols,
    feature_names=feature_cols
)

pa_model_v2 = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=400,
    depth=8,
    learning_rate=0.08,
    random_seed=RANDOM_SEED,
    verbose=50,
    task_type="GPU"
)

pa_model_v2.fit(train_pool, eval_set=test_pool, use_best_model=True)

print("training complete")
print("classes:", pa_model_v2.classes_)

X_train shape: (781235, 21)
X_test shape: (192135, 21)
Duplicate X_train columns: []
0:	learn: 1.7744389	test: 1.7785561	best: 1.7785561 (0)	total: 40.4ms	remaining: 16.1s
50:	learn: 1.0060748	test: 1.0235481	best: 1.0235481 (50)	total: 2s	remaining: 13.7s
100:	learn: 0.9914149	test: 1.0144249	best: 1.0144249 (100)	total: 4.09s	remaining: 12.1s
150:	learn: 0.9839613	test: 1.0106423	best: 1.0106423 (150)	total: 5.75s	remaining: 9.48s
200:	learn: 0.9786408	test: 1.0091181	best: 1.0091181 (200)	total: 7.51s	remaining: 7.44s
250:	learn: 0.9741857	test: 1.0077955	best: 1.0077868 (249)	total: 9.24s	remaining: 5.49s
300:	learn: 0.9705254	test: 1.0071846	best: 1.0071846 (300)	total: 11s	remaining: 3.61s
350:	learn: 0.9668914	test: 1.0064740	best: 1.0064498 (347)	total: 12.6s	remaining: 1.76s
399:	learn: 0.9640335	test: 1.0062528	best: 1.0061357 (379)	total: 14.2s	remaining: 0us
bestTest = 1.00613566
bestIteration = 379
Shrink model to first 380 iterations.
training complete
classes: ['double' 

In [18]:
# ----------------------------
# Evaluate PA model on 2025 plate appearances
# ----------------------------

test_pred_proba = pa_model_v2.predict_proba(X_test)
test_pred_class = pa_model_v2.predict(X_test)

test_pred_class = np.array(test_pred_class).reshape(-1)

pa_accuracy = (test_pred_class == y_test.values).mean()

print("PA classification accuracy:", round(pa_accuracy, 4))

proba_cols = list(pa_model_v2.classes_)
proba_df = pd.DataFrame(test_pred_proba, columns=proba_cols, index=X_test.index)

print("\nAverage predicted probabilities by class:")
print(proba_df.mean().sort_values(ascending=False))

display(proba_df.head())

PA classification accuracy: 0.6161

Average predicted probabilities by class:
out          0.449266
strikeout    0.240975
single       0.145116
walk         0.085062
double       0.043566
home_run     0.032085
triple       0.003929
dtype: float64


,double,home_run,out,single,strikeout,triple,walk
3075349,0.048645,0.018171,0.390734,0.217418,0.319157,0.005813,0.000062
3075352,0.002732,0.001740,0.012336,0.005599,0.000011,0.001033,0.976548
3075357,0.009531,0.008792,0.130006,0.033383,0.168801,0.000861,0.648625
3075363,0.069460,0.081451,0.499577,0.151387,0.187372,0.004665,0.006088
3075370,0.083951,0.084823,0.624634,0.202452,0.000004,0.003699,0.000436


In [19]:
# ----------------------------
# Save rebuilt PA model + metadata
# ----------------------------

os.makedirs("models", exist_ok=True)

SAVE_MODEL_PATH = "models/winloss_pa_model.cbm"
SAVE_META_PATH = "models/winloss_pa_model_meta.json"

pa_model_v2.save_model(SAVE_MODEL_PATH)

save_meta = {
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "target_col": "pa_outcome",
    "train_years": TRAIN_YEARS,
    "test_years": TEST_YEARS,
    "classes": list(pa_model_v2.classes_)
}

with open(SAVE_META_PATH, "w") as f:
    json.dump(save_meta, f, indent=2)

print("saved model to:", SAVE_MODEL_PATH)
print("saved metadata to:", SAVE_META_PATH)

saved model to: models/winloss_pa_model.cbm
saved metadata to: models/winloss_pa_model_meta.json


In [20]:
# ----------------------------
# Build 2025 game summary table from raw Statcast
# ----------------------------

raw["game_date"] = pd.to_datetime(raw["game_date"], errors="coerce")
raw_2025 = raw[raw["game_date"].dt.year == 2025].copy()

game_level = (
    raw_2025.groupby("game_pk", as_index=False)
    .agg({
        "game_date": "first",
        "home_team": "first",
        "away_team": "first",
        "home_score": "max",
        "away_score": "max"
    })
    .sort_values(["game_date", "game_pk"])
    .reset_index(drop=True)
)

game_level["home_win"] = (game_level["home_score"] > game_level["away_score"]).astype(int)
game_level["away_win"] = (game_level["away_score"] > game_level["home_score"]).astype(int)
game_level["run_diff"] = game_level["home_score"] - game_level["away_score"]

print("Number of 2025 games:", len(game_level))
display(game_level.head(10))

Number of 2025 games: 2626


,game_pk,game_date,home_team,away_team,home_score,away_score,home_win,away_win,run_diff
0,778698,2025-03-15,ATL,MIN,0,4,0,1,-4
1,778718,2025-03-15,HOU,MIA,6,4,1,0,2
2,778719,2025-03-15,WSH,NYM,4,1,1,0,3
3,778766,2025-03-15,BOS,ATL,3,7,0,1,-4
4,778787,2025-03-15,AZ,CWS,7,8,0,1,-1
5,778819,2025-03-15,CIN,KC,7,13,0,1,-6
6,778844,2025-03-15,CWS,COL,1,1,0,0,0
7,778906,2025-03-15,PHI,DET,2,2,0,0,0
8,778920,2025-03-15,KC,CLE,5,7,0,1,-2
9,778968,2025-03-15,TB,NYY,7,7,0,0,0


In [21]:
# ----------------------------
# Approximate starting lineups from early game plate appearances
# ----------------------------

pa_2025 = pa_df[pa_df["game_date"].dt.year == 2025].copy()

# keep only early innings to approximate starting lineups
early_pa = pa_2025[pa_2025["inning"] <= 3].copy()

# order plate appearances within each half-inning context
sort_cols = ["game_pk", "inning", "inning_topbot"]
if "at_bat_number" in early_pa.columns:
    sort_cols.append("at_bat_number")
elif "pitch_number" in early_pa.columns:
    sort_cols.append("pitch_number")

early_pa = early_pa.sort_values(sort_cols).copy()

# first appearance order for each batter in a game side
lineup_approx = (
    early_pa.groupby(["game_pk", "inning_topbot", "batter"], as_index=False)
    .agg({
        "inning": "min"
    })
    .sort_values(["game_pk", "inning_topbot", "inning"])
    .copy()
)

# assign lineup slot by appearance order within game side
lineup_approx["lineup_slot"] = (
    lineup_approx.groupby(["game_pk", "inning_topbot"]).cumcount() + 1
)

# keep top 9
lineup_approx = lineup_approx[lineup_approx["lineup_slot"] <= 9].copy()

print("Approx lineup rows:", len(lineup_approx))
display(lineup_approx.head(20))

Approx lineup rows: 46664


,game_pk,inning_topbot,batter,inning,lineup_slot
0,776135,Bot,545361,1,1
1,776135,Bot,621035,1,2
2,776135,Bot,650859,1,3
3,776135,Bot,666176,1,4
4,776135,Bot,666198,2,5
5,776135,Bot,669326,2,6
7,776135,Bot,694203,2,7
8,776135,Bot,695681,2,8
6,776135,Bot,681351,3,9
9,776135,Top,602104,1,1


In [22]:
# ----------------------------
# Build pregame matchup table
# ----------------------------

# likely starter = pitcher with earliest first appearance for each defensive side
pitcher_cols = ["game_pk", "inning_topbot", "pitcher", "p_throws"]
pitcher_source = pa_2025[pitcher_cols + ["inning"]].copy()

starter_pitchers = (
    pitcher_source.sort_values(["game_pk", "inning_topbot", "inning"])
    .groupby(["game_pk", "inning_topbot"], as_index=False)
    .first()
)

# split home batting side vs away batting side
away_lineup = lineup_approx[lineup_approx["inning_topbot"] == "Top"].copy()
home_lineup = lineup_approx[lineup_approx["inning_topbot"] == "Bot"].copy()

away_starter = starter_pitchers[starter_pitchers["inning_topbot"] == "Bot"].copy()
home_starter = starter_pitchers[starter_pitchers["inning_topbot"] == "Top"].copy()

away_starter = away_starter.rename(columns={
    "pitcher": "opp_pitcher",
    "p_throws": "opp_p_throws"
})[["game_pk", "opp_pitcher", "opp_p_throws"]]

home_starter = home_starter.rename(columns={
    "pitcher": "opp_pitcher",
    "p_throws": "opp_p_throws"
})[["game_pk", "opp_pitcher", "opp_p_throws"]]

away_matchups = away_lineup.merge(away_starter, on="game_pk", how="left")
home_matchups = home_lineup.merge(home_starter, on="game_pk", how="left")

away_matchups["batting_team_side"] = "away"
home_matchups["batting_team_side"] = "home"

pregame_matchups = pd.concat([away_matchups, home_matchups], ignore_index=True)

# get batter handedness
batter_hand = (
    pa_2025.groupby("batter", as_index=False)
    .agg({"stand": lambda s: s.dropna().mode().iloc[0] if len(s.dropna()) else "UNK"})
)

pregame_matchups = pregame_matchups.merge(batter_hand, on="batter", how="left")

pregame_matchups = pregame_matchups.merge(
    game_level[["game_pk", "game_date", "home_team", "away_team", "home_score", "away_score", "home_win"]],
    on="game_pk",
    how="left"
)

print("Pregame matchup rows:", len(pregame_matchups))
display(pregame_matchups.head(20))

Pregame matchup rows: 46664


,game_pk,inning_topbot,batter,inning,lineup_slot,opp_pitcher,opp_p_throws,batting_team_side,stand,game_date,home_team,away_team,home_score,away_score,home_win
0,776135,Top,602104,1,1,621121,R,away,R,2025-09-28,LAA,HOU,2,6,0
1,776135,Top,643289,1,2,621121,R,away,R,2025-09-28,LAA,HOU,2,6,0
2,776135,Top,660821,1,3,621121,R,away,L,2025-09-28,LAA,HOU,2,6,0
3,776135,Top,673237,1,4,621121,R,away,R,2025-09-28,LAA,HOU,2,6,0
4,776135,Top,805904,1,5,621121,R,away,L,2025-09-28,LAA,HOU,2,6,0
5,776135,Top,605170,2,6,621121,R,away,L,2025-09-28,LAA,HOU,2,6,0
6,776135,Top,689200,2,7,621121,R,away,L,2025-09-28,LAA,HOU,2,6,0
7,776135,Top,694728,2,8,621121,R,away,R,2025-09-28,LAA,HOU,2,6,0
8,776135,Top,701358,3,9,621121,R,away,R,2025-09-28,LAA,HOU,2,6,0
9,776136,Top,606466,1,1,694297,R,away,L,2025-09-28,SD,AZ,12,4,1


In [23]:
# ----------------------------
# Build pregame feature rows for lineup vs starter
# ----------------------------

base_defaults = {
    "balls": 0,
    "strikes": 0,
    "outs_when_up": 0,
    "inning": 1,
    "inning_topbot": "Top",
    "on_1b": np.nan,
    "on_2b": np.nan,
    "on_3b": np.nan,
    "bat_score": 0,
    "fld_score": 0,
    "release_speed": np.nan,
    "release_spin_rate": np.nan,
    "zone": np.nan,
    "pfx_x": np.nan,
    "pfx_z": np.nan
}

pregame_rows = pregame_matchups.copy()

for col, val in base_defaults.items():
    if col not in pregame_rows.columns:
        pregame_rows[col] = val
    else:
        pregame_rows[col] = pregame_rows[col].fillna(val)

pregame_rows["inning_topbot"] = np.where(
    pregame_rows["batting_team_side"] == "away",
    "Top",
    "Bot"
)

pregame_rows["pitcher"] = pregame_rows["opp_pitcher"]
pregame_rows["p_throws"] = pregame_rows["opp_p_throws"]

# set teams correctly
pregame_rows["batting_team"] = np.where(
    pregame_rows["batting_team_side"] == "away",
    pregame_rows["away_team"],
    pregame_rows["home_team"]
)

pregame_rows["fielding_team"] = np.where(
    pregame_rows["batting_team_side"] == "away",
    pregame_rows["home_team"],
    pregame_rows["away_team"]
)

# keep only columns needed for model prediction
for c in feature_cols:
    if c not in pregame_rows.columns:
        pregame_rows[c] = np.nan

# fill numeric from train medians
train_feature_medians = {}
for c in feature_cols:
    if c not in cat_cols:
        med = train_pa[c].median()
        train_feature_medians[c] = med
        pregame_rows[c] = pd.to_numeric(pregame_rows[c], errors="coerce").fillna(med)

# fill categoricals
for c in cat_cols:
    pregame_rows[c] = pregame_rows[c].astype("string").fillna("UNK").astype(str)

print("Pregame rows shape:", pregame_rows.shape)
display(pregame_rows[[
    "game_pk", "batting_team_side", "lineup_slot", "batter", "pitcher", "stand", "p_throws"
] + [c for c in ["home_team", "away_team", "balls", "strikes", "outs_when_up", "inning"] if c in pregame_rows.columns]].head(20))

Pregame rows shape: (46664, 32)


,game_pk,batting_team_side,lineup_slot,batter,pitcher,stand,p_throws,home_team,away_team,balls,strikes,outs_when_up,inning
0,776135,away,1,602104,621121,R,R,LAA,HOU,0,0,0,1
1,776135,away,2,643289,621121,R,R,LAA,HOU,0,0,0,1
2,776135,away,3,660821,621121,L,R,LAA,HOU,0,0,0,1
3,776135,away,4,673237,621121,R,R,LAA,HOU,0,0,0,1
4,776135,away,5,805904,621121,L,R,LAA,HOU,0,0,0,1
5,776135,away,6,605170,621121,L,R,LAA,HOU,0,0,0,2
6,776135,away,7,689200,621121,L,R,LAA,HOU,0,0,0,2
7,776135,away,8,694728,621121,R,R,LAA,HOU,0,0,0,2
8,776135,away,9,701358,621121,R,R,LAA,HOU,0,0,0,3
9,776136,away,1,606466,694297,L,R,SD,AZ,0,0,0,1


In [27]:
# ----------------------------
# Predict PA outcome probabilities for each lineup batter vs starter
# ----------------------------

pregame_X = pregame_rows[feature_cols].copy()
print("pregame_X shape:", pregame_X.shape)

pregame_pred_proba = pa_model_v2.predict_proba(pregame_X)

pregame_prob_df = pd.DataFrame(
    pregame_pred_proba,
    columns=list(pa_model_v2.classes_),
    index=pregame_rows.index
)

base_info_cols = [
    "game_pk", "game_date", "home_team", "away_team",
    "batting_team_side", "lineup_slot", "batter", "pitcher", "home_win"
]

pregame_probs = pd.concat(
    [
        pregame_rows[base_info_cols].reset_index(drop=True),
        pregame_prob_df.reset_index(drop=True)
    ],
    axis=1
)

print("Pregame probability table shape:", pregame_probs.shape)
print("Pregame probability columns:")
print(pregame_probs.columns.tolist())

display(pregame_probs.head(20))

pregame_X shape: (46664, 21)
Pregame probability table shape: (46664, 16)
Pregame probability columns:
['game_pk', 'game_date', 'home_team', 'away_team', 'batting_team_side', 'lineup_slot', 'batter', 'pitcher', 'home_win', 'double', 'home_run', 'out', 'single', 'strikeout', 'triple', 'walk']


,game_pk,game_date,home_team,away_team,batting_team_side,lineup_slot,batter,pitcher,home_win,double,home_run,out,single,strikeout,triple,walk
0,776135,2025-09-28,LAA,HOU,away,1,602104,621121,0,0.082009,0.037912,0.658495,0.216075,0.000002,0.005422,0.000085
1,776135,2025-09-28,LAA,HOU,away,2,643289,621121,0,0.076729,0.024129,0.677310,0.217046,0.000002,0.004689,0.000095
2,776135,2025-09-28,LAA,HOU,away,3,660821,621121,0,0.074040,0.029384,0.666013,0.223400,0.000002,0.007078,0.000084
3,776135,2025-09-28,LAA,HOU,away,4,673237,621121,0,0.072283,0.026029,0.683296,0.212896,0.000002,0.005406,0.000088
4,776135,2025-09-28,LAA,HOU,away,5,805904,621121,0,0.055896,0.015710,0.692279,0.227605,0.000004,0.008407,0.000100
5,776135,2025-09-28,LAA,HOU,away,6,605170,621121,0,0.072516,0.021940,0.675919,0.223144,0.000001,0.006385,0.000095
6,776135,2025-09-28,LAA,HOU,away,7,689200,621121,0,0.059664,0.014273,0.689410,0.230922,0.000002,0.005630,0.000098
7,776135,2025-09-28,LAA,HOU,away,8,694728,621121,0,0.067624,0.019794,0.678050,0.229104,0.000005,0.005341,0.000083
8,776135,2025-09-28,LAA,HOU,away,9,701358,621121,0,0.064621,0.018772,0.680461,0.230842,0.000004,0.005215,0.000085
9,776136,2025-09-28,SD,AZ,away,1,606466,694297,1,0.070609,0.024826,0.670821,0.228502,0.000001,0.005158,0.000083


In [28]:
# ----------------------------
# Build simple expected runs proxy from lineup PA probabilities
# ----------------------------

if "pregame_probs" not in globals():
    raise ValueError("pregame_probs does not exist. Run Cell 20 first.")

run_weights = {
    "out": 0.00,
    "strikeout": 0.00,
    "walk": 0.30,
    "single": 0.47,
    "double": 0.77,
    "triple": 1.04,
    "home_run": 1.40
}

pregame_probs = pregame_probs.copy()

for outcome, weight in run_weights.items():
    if outcome in pregame_probs.columns:
        pregame_probs[f"rw_{outcome}"] = pregame_probs[outcome] * weight
    else:
        print(f"warning: {outcome} not found in pregame_probs")

rw_cols = [c for c in pregame_probs.columns if c.startswith("rw_")]
print("run-weight columns:", rw_cols)

pregame_probs["expected_pa_value"] = pregame_probs[rw_cols].sum(axis=1)

team_strength = (
    pregame_probs.groupby(
        ["game_pk", "game_date", "home_team", "away_team", "batting_team_side", "home_win"],
        as_index=False
    )["expected_pa_value"]
    .mean()
)

strength_pivot = (
    team_strength.pivot_table(
        index=["game_pk", "game_date", "home_team", "away_team", "home_win"],
        columns="batting_team_side",
        values="expected_pa_value"
    )
    .reset_index()
)

strength_pivot.columns.name = None
strength_pivot = strength_pivot.rename(columns={
    "away": "away_strength",
    "home": "home_strength"
})

strength_pivot["home_strength"] = strength_pivot["home_strength"].fillna(strength_pivot["home_strength"].median())
strength_pivot["away_strength"] = strength_pivot["away_strength"].fillna(strength_pivot["away_strength"].median())

strength_pivot["home_strength_share"] = (
    strength_pivot["home_strength"] /
    (strength_pivot["home_strength"] + strength_pivot["away_strength"])
)

print("Historical game strength table:", strength_pivot.shape)
display(strength_pivot.head(20))

run-weight columns: ['rw_out', 'rw_strikeout', 'rw_walk', 'rw_single', 'rw_double', 'rw_triple', 'rw_home_run']
Historical game strength table: (2626, 8)


,game_pk,game_date,home_team,away_team,home_win,away_strength,home_strength,home_strength_share
0,776135,2025-09-28,LAA,HOU,0,0.197090,0.236718,0.545675
1,776136,2025-09-28,SD,AZ,1,0.195921,0.214307,0.522410
2,776137,2025-09-28,SF,COL,1,0.219802,0.211972,0.490932
3,776138,2025-09-28,ATH,KC,0,0.196514,0.213678,0.520922
4,776139,2025-09-28,SEA,LAD,0,0.202430,0.215006,0.515062
5,776140,2025-09-28,NYY,BAL,1,0.213037,0.229285,0.518366
6,776141,2025-09-28,PHI,MIN,0,0.209052,0.231499,0.525476
7,776142,2025-09-28,CHC,STL,1,0.208421,0.217849,0.511058
8,776143,2025-09-28,WSH,CWS,0,0.203955,0.224191,0.523632
9,776144,2025-09-28,MIL,CIN,1,0.206304,0.205038,0.498462


In [29]:
# ----------------------------
# Evaluate pregame winner prediction
# ----------------------------

eval_df = strength_pivot.copy()

eval_df["pred_home_win"] = (eval_df["home_strength_share"] >= 0.5).astype(int)

accuracy = (eval_df["pred_home_win"] == eval_df["home_win"]).mean()
brier = brier_score_loss(eval_df["home_win"], eval_df["home_strength_share"])
ll = log_loss(
    eval_df["home_win"],
    np.clip(eval_df["home_strength_share"], 1e-6, 1 - 1e-6)
)

print("Historical pregame home-win accuracy:", round(accuracy, 4))
print("Historical pregame Brier score:", round(brier, 4))
print("Historical pregame log loss:", round(ll, 4))

print("\nPredicted home win rate:", round(eval_df["pred_home_win"].mean(), 4))
print("Actual home win rate:", round(eval_df["home_win"].mean(), 4))
print("Average predicted home strength share:", round(eval_df["home_strength_share"].mean(), 4))

display(eval_df.head(20))

Historical pregame home-win accuracy: 0.4905
Historical pregame Brier score: 0.2508
Historical pregame log loss: 0.6947

Predicted home win rate: 0.6588
Actual home win rate: 0.4501
Average predicted home strength share: 0.5063


,game_pk,game_date,home_team,away_team,home_win,away_strength,home_strength,home_strength_share,pred_home_win
0,776135,2025-09-28,LAA,HOU,0,0.197090,0.236718,0.545675,1
1,776136,2025-09-28,SD,AZ,1,0.195921,0.214307,0.522410,1
2,776137,2025-09-28,SF,COL,1,0.219802,0.211972,0.490932,0
3,776138,2025-09-28,ATH,KC,0,0.196514,0.213678,0.520922,1
4,776139,2025-09-28,SEA,LAD,0,0.202430,0.215006,0.515062,1
5,776140,2025-09-28,NYY,BAL,1,0.213037,0.229285,0.518366,1
6,776141,2025-09-28,PHI,MIN,0,0.209052,0.231499,0.525476,1
7,776142,2025-09-28,CHC,STL,1,0.208421,0.217849,0.511058,1
8,776143,2025-09-28,WSH,CWS,0,0.203955,0.224191,0.523632,1
9,776144,2025-09-28,MIL,CIN,1,0.206304,0.205038,0.498462,0


In [55]:
# ----------------------------
# Build starter + team-specific bullpen probability caches
# ----------------------------

if "pregame_probs_complete" not in globals():
    raise ValueError("Run the updated Cell 24 first so pregame_probs_complete exists.")

OUTCOME_COLS = ["double", "home_run", "out", "single", "strikeout", "triple", "walk"]
OUTCOME_COLS = [c for c in OUTCOME_COLS if c in pregame_probs_complete.columns]

# --------------------------------------------------
# 1) Starter cache from pregame batter vs starter rows
# --------------------------------------------------
starter_cache = {}

for _, row in pregame_probs_complete.iterrows():
    key = (
        int(row["game_pk"]),
        row["batting_team_side"],
        int(row["lineup_slot"])
    )
    probs = row[OUTCOME_COLS].astype(float).values
    probs = probs / probs.sum()
    starter_cache[key] = probs

print("Starter cache entries:", len(starter_cache))
print("Outcome order:", OUTCOME_COLS)

# --------------------------------------------------
# 2) Game -> home/away team lookup
# --------------------------------------------------
game_team_lookup = (
    pregame_probs_complete[["game_pk", "home_team", "away_team"]]
    .drop_duplicates()
    .copy()
)

game_team_lookup["game_pk"] = game_team_lookup["game_pk"].astype(int)

game_team_lookup = {
    int(row["game_pk"]): {
        "home_team": row["home_team"],
        "away_team": row["away_team"]
    }
    for _, row in game_team_lookup.iterrows()
}

print("Game/team lookup entries:", len(game_team_lookup))

# --------------------------------------------------
# 3) Build team-specific bullpen distributions
#    We approximate bullpen as late-inning defense.
# --------------------------------------------------
bullpen_source = pa_2025.copy()

# late innings only
bullpen_source = bullpen_source[bullpen_source["inning"] >= 6].copy()

# who is fielding / defending?
bullpen_source["defending_team"] = np.where(
    bullpen_source["inning_topbot"] == "Top",
    bullpen_source["home_team"],
    bullpen_source["away_team"]
)

# count PA outcomes by defending team
team_bp_counts = (
    bullpen_source.groupby(["defending_team", "pa_outcome"])
    .size()
    .reset_index(name="n")
)

team_bp_counts["total"] = team_bp_counts.groupby("defending_team")["n"].transform("sum")
team_bp_counts["p"] = team_bp_counts["n"] / team_bp_counts["total"]

# global fallback
global_bp_counts = (
    bullpen_source.groupby("pa_outcome")
    .size()
    .reset_index(name="n")
)
global_bp_counts["p"] = global_bp_counts["n"] / global_bp_counts["n"].sum()

team_bullpen_cache = {}

for team in sorted(team_bp_counts["defending_team"].dropna().unique().tolist()):
    sub = team_bp_counts[team_bp_counts["defending_team"] == team].copy()

    probs = []
    for outcome in OUTCOME_COLS:
        match = sub.loc[sub["pa_outcome"] == outcome, "p"]
        probs.append(float(match.iloc[0]) if len(match) else 0.0)

    probs = np.array(probs, dtype=float)

    if probs.sum() == 0:
        # fallback to global bullpen distribution
        probs = []
        for outcome in OUTCOME_COLS:
            match = global_bp_counts.loc[global_bp_counts["pa_outcome"] == outcome, "p"]
            probs.append(float(match.iloc[0]) if len(match) else 0.0)
        probs = np.array(probs, dtype=float)

    probs = probs / probs.sum()
    team_bullpen_cache[team] = probs

print("Team bullpen cache entries:", len(team_bullpen_cache))

# sample outputs
sample_starter_keys = list(starter_cache.keys())[:3]
print("\nSample starter cache:")
for k in sample_starter_keys:
    print(k, starter_cache[k])

sample_bp_teams = list(team_bullpen_cache.keys())[:5]
print("\nSample team bullpen cache:")
for team in sample_bp_teams:
    print(team, team_bullpen_cache[team])

Starter cache entries: 37692
Outcome order: ['double', 'home_run', 'out', 'single', 'strikeout', 'triple', 'walk']
Game/team lookup entries: 2094
Team bullpen cache entries: 30

Sample starter cache:
(776135, 'away', 1) [8.20091147e-02 3.79118246e-02 6.58494636e-01 2.16075264e-01
 1.72142167e-06 5.42217342e-03 8.52661421e-05]
(776135, 'away', 2) [7.67285871e-02 2.41291372e-02 6.77310119e-01 2.17045705e-01
 2.38144971e-06 4.68949628e-03 9.45732944e-05]
(776135, 'away', 3) [7.40399074e-02 2.93843250e-02 6.66012954e-01 2.23399968e-01
 1.52645679e-06 7.07773287e-03 8.35867502e-05]

Sample team bullpen cache:
ATH [0.0453227  0.02936911 0.44742567 0.13959391 0.22733865 0.00507614
 0.10587382]
ATL [0.03995601 0.0318915  0.46151026 0.13526393 0.23753666 0.00219941
 0.09164223]
AZ [0.05478018 0.03070482 0.4542917  0.14096301 0.2166783  0.00523378
 0.09734822]
BAL [0.04169688 0.02791878 0.43255983 0.15083394 0.24583031 0.00290065
 0.09825961]
BOS [0.03478878 0.0227192  0.4600639  0.15654952 0.23

In [39]:
# ----------------------------
# Build lineup order lookup and keep only complete 9-man lineups
# ----------------------------

# count lineup slots by game and side
lineup_counts = (
    pregame_probs.groupby(["game_pk", "batting_team_side"])["lineup_slot"]
    .nunique()
    .reset_index(name="n_slots")
)

complete_games = (
    lineup_counts.pivot(index="game_pk", columns="batting_team_side", values="n_slots")
    .fillna(0)
    .reset_index()
)

complete_games = complete_games[
    (complete_games["away"] == 9) &
    (complete_games["home"] == 9)
].copy()

complete_game_pks = set(complete_games["game_pk"].astype(int).tolist())

print("Complete games with full 9-man lineups on both sides:", len(complete_game_pks))

# filter pregame_probs to complete games only
pregame_probs_complete = pregame_probs[
    pregame_probs["game_pk"].astype(int).isin(complete_game_pks)
].copy()

# optional: make sure every side has slots 1-9 exactly
slot_check = (
    pregame_probs_complete.groupby(["game_pk", "batting_team_side"])["lineup_slot"]
    .apply(lambda s: sorted(s.astype(int).unique().tolist()))
    .reset_index(name="slots")
)

print("\nSample slot check:")
display(slot_check.head(10))

lineup_lookup = {}
for _, row in pregame_probs_complete.iterrows():
    game_pk = int(row["game_pk"])
    side = row["batting_team_side"]
    slot = int(row["lineup_slot"])
    batter = row["batter"]
    lineup_lookup[(game_pk, side, slot)] = batter

print("Lineup lookup entries:", len(lineup_lookup))
display(
    pregame_probs_complete[["game_pk", "batting_team_side", "lineup_slot", "batter"]]
    .sort_values(["game_pk", "batting_team_side", "lineup_slot"])
    .head(18)
)
display(sample_lineups)

Complete games with full 9-man lineups on both sides: 2094

Sample slot check:


,game_pk,batting_team_side,slots
0,776135,away,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
1,776135,home,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
2,776137,away,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
3,776137,home,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
4,776138,away,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
5,776138,home,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
6,776139,away,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
7,776139,home,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
8,776140,away,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"
9,776140,home,"[1, 2, 3, 4, 5, 6, 7, 8, 9]"


Lineup lookup entries: 37692


,game_pk,batting_team_side,lineup_slot,batter
0,776135,away,1,602104
1,776135,away,2,643289
2,776135,away,3,660821
3,776135,away,4,673237
4,776135,away,5,805904
5,776135,away,6,605170
6,776135,away,7,689200
7,776135,away,8,694728
8,776135,away,9,701358
23337,776135,home,1,545361


,game_pk,batting_team_side,lineup_slot,batter
0,776135,away,1,602104
1,776135,away,2,643289
2,776135,away,3,660821
3,776135,away,4,673237
4,776135,away,5,805904
5,776135,away,6,605170
6,776135,away,7,689200
7,776135,away,8,694728
8,776135,away,9,701358
9,776136,away,1,606466


In [56]:
# ----------------------------
# Game simulation helper functions
# ----------------------------

BULLPEN_START_INNING = 6

def get_defending_team(game_pk, batting_side):
    teams = game_team_lookup[int(game_pk)]
    if batting_side == "away":
        return teams["home_team"]
    elif batting_side == "home":
        return teams["away_team"]
    else:
        raise ValueError(f"Unknown batting_side: {batting_side}")


def sample_pa_outcome(game_pk, batting_side, lineup_slot, inning, rng):
    starter_key = (int(game_pk), batting_side, int(lineup_slot))

    if inning < BULLPEN_START_INNING and starter_key in starter_cache:
        probs = starter_cache[starter_key]
    else:
        defending_team = get_defending_team(game_pk, batting_side)

        if defending_team in team_bullpen_cache:
            probs = team_bullpen_cache[defending_team]
        elif starter_key in starter_cache:
            probs = starter_cache[starter_key]
        else:
            raise KeyError(f"Missing cache for {(game_pk, batting_side, lineup_slot, inning)}")

    return rng.choice(OUTCOME_COLS, p=probs)


def advance_runners(event, bases, outs, runs):
    """
    bases = [on_1b, on_2b, on_3b] as 0/1 flags
    returns updated (bases, outs, runs)
    """
    on_1b, on_2b, on_3b = bases

    if event in ["out", "strikeout"]:
        outs += 1
        return [on_1b, on_2b, on_3b], outs, runs

    if event == "walk":
        if on_1b and on_2b and on_3b:
            runs += 1
            return [1, 1, 1], outs, runs
        if on_1b and on_2b:
            return [1, 1, 1], outs, runs
        if on_1b:
            return [1, 1, on_3b], outs, runs
        return [1, on_2b, on_3b], outs, runs

    if event == "single":
        runs += on_3b
        new_on_3b = on_2b
        new_on_2b = on_1b
        new_on_1b = 1
        return [new_on_1b, new_on_2b, new_on_3b], outs, runs

    if event == "double":
        runs += on_3b + on_2b
        new_on_3b = on_1b
        new_on_2b = 1
        new_on_1b = 0
        return [new_on_1b, new_on_2b, new_on_3b], outs, runs

    if event == "triple":
        runs += on_1b + on_2b + on_3b
        return [0, 0, 1], outs, runs

    if event == "home_run":
        runs += on_1b + on_2b + on_3b + 1
        return [0, 0, 0], outs, runs

    outs += 1
    return [on_1b, on_2b, on_3b], outs, runs

In [57]:
# ----------------------------
# Simulate one half inning
# ----------------------------

def simulate_half_inning(game_pk, batting_side, lineup_slot_start, inning, rng):
    outs = 0
    runs = 0
    bases = [0, 0, 0]
    lineup_slot = lineup_slot_start

    while outs < 3:
        event = sample_pa_outcome(game_pk, batting_side, lineup_slot, inning, rng)
        bases, outs, runs = advance_runners(event, bases, outs, runs)

        lineup_slot += 1
        if lineup_slot > 9:
            lineup_slot = 1

    return runs, lineup_slot

In [58]:
# ----------------------------
# Simulate one full game
# ----------------------------

def simulate_game(game_pk, rng, max_innings=9):
    away_runs = 0
    home_runs = 0

    away_slot = 1
    home_slot = 1

    inning = 1

    while inning <= max_innings:
        # top half
        runs_scored, away_slot = simulate_half_inning(game_pk, "away", away_slot, inning, rng)
        away_runs += runs_scored

        # if bottom 9+ and home already leads, game ends
        if inning >= 9 and home_runs > away_runs:
            break

        # bottom half
        runs_scored, home_slot = simulate_half_inning(game_pk, "home", home_slot, inning, rng)
        home_runs += runs_scored

        # walk-off: if bottom 9+ ends with home team ahead, game ends
        if inning >= 9 and home_runs > away_runs:
            break

        inning += 1

    while away_runs == home_runs:
        inning += 1

        runs_scored, away_slot = simulate_half_inning(game_pk, "away", away_slot, inning, rng)
        away_runs += runs_scored

        if home_runs > away_runs:
            break

        runs_scored, home_slot = simulate_half_inning(game_pk, "home", home_slot, inning, rng)
        home_runs += runs_scored

        if home_runs > away_runs:
            break

    return {
        "game_pk": int(game_pk),
        "away_runs": int(away_runs),
        "home_runs": int(home_runs),
        "home_win": int(home_runs > away_runs)
    }

In [44]:
# ----------------------------
# Quick test on one game
# ----------------------------

test_game_pk = int(strength_pivot["game_pk"].iloc[0])
rng = np.random.default_rng(42)

test_results = [simulate_game(test_game_pk, rng) for _ in range(10)]
test_results_df = pd.DataFrame(test_results)

print("Test game_pk:", test_game_pk)
display(test_results_df)

print("Avg away runs:", round(test_results_df["away_runs"].mean(), 2))
print("Avg home runs:", round(test_results_df["home_runs"].mean(), 2))
print("Home win pct:", round(test_results_df["home_win"].mean(), 4))

Test game_pk: 776135


,game_pk,away_runs,home_runs,home_win
0,776135,2,3,1
1,776135,6,5,0
2,776135,2,7,1
3,776135,7,5,0
4,776135,5,6,1
5,776135,6,5,0
6,776135,2,5,1
7,776135,8,5,0
8,776135,4,2,0
9,776135,0,9,1


Avg away runs: 4.2
Avg home runs: 5.2
Home win pct: 0.5


In [61]:
# ----------------------------
# Run improved Monte Carlo simulation for historical games
# ----------------------------

def simulate_many_games(game_pk, n_sims=2000, seed=42):
    rng = np.random.default_rng(seed)
    results = [simulate_game(game_pk, rng) for _ in range(n_sims)]
    df = pd.DataFrame(results)

    return {
        "game_pk": int(game_pk),
        "sim_home_runs_mean": df["home_runs"].mean(),
        "sim_away_runs_mean": df["away_runs"].mean(),
        "sim_home_win_prob": df["home_win"].mean(),
        "proj_home_runs": int(np.rint(df["home_runs"].mean())),
        "proj_away_runs": int(np.rint(df["away_runs"].mean()))
    }

sample_game_pks = sorted(list(complete_game_pks))[:200]

sim_summaries = []
failed_games = []

for i, game_pk in enumerate(sample_game_pks, start=1):
    try:
        sim_summaries.append(simulate_many_games(game_pk, n_sims=2000, seed=42 + i))
    except Exception as e:
        failed_games.append((game_pk, str(e)))

sim_summary_df = pd.DataFrame(sim_summaries)

display(sim_summary_df.head(10))
print("Finished simming", len(sim_summary_df), "games")
print("Failed games:", failed_games[:10])

,game_pk,sim_home_runs_mean,sim_away_runs_mean,sim_home_win_prob,proj_home_runs,proj_away_runs
0,776135,4.5460,3.9695,0.5660,5,4
1,776137,4.5740,4.2920,0.5460,5,4
2,776138,4.0090,3.6745,0.5575,4,4
3,776139,4.1745,3.8000,0.5535,4,4
4,776140,4.6010,4.3770,0.5320,5,4
5,776141,4.5785,3.9940,0.5730,5,4
6,776142,4.0485,4.0750,0.5050,4,4
7,776143,4.6575,4.2895,0.5365,5,4
8,776144,3.7670,3.8245,0.5120,4,4
9,776145,3.8160,3.7355,0.5190,4,4


Finished simming 200 games
Failed games: []


In [63]:
# ----------------------------
# Evaluate Monte Carlo game predictions
# ----------------------------

mc_eval = sim_summary_df.merge(
    game_level[["game_pk", "game_date", "home_team", "away_team", "home_score", "away_score", "home_win"]],
    on="game_pk",
    how="left"
)

mc_eval["pred_home_win"] = (mc_eval["sim_home_win_prob"] >= 0.5).astype(int)

mc_accuracy = (mc_eval["pred_home_win"] == mc_eval["home_win"]).mean()
mc_brier = brier_score_loss(mc_eval["home_win"], mc_eval["sim_home_win_prob"])
mc_logloss = log_loss(
    mc_eval["home_win"],
    np.clip(mc_eval["sim_home_win_prob"], 1e-6, 1 - 1e-6)
)

print("Monte Carlo home-win accuracy:", round(mc_accuracy, 4))
print("Monte Carlo Brier score:", round(mc_brier, 4))
print("Monte Carlo log loss:", round(mc_logloss, 4))

display(mc_eval.head(20))

Monte Carlo home-win accuracy: 0.505
Monte Carlo Brier score: 0.2512
Monte Carlo log loss: 0.6955


,game_pk,sim_home_runs_mean,sim_away_runs_mean,sim_home_win_prob,proj_home_runs,proj_away_runs,game_date,home_team,away_team,home_score,away_score,home_win,pred_home_win
0,776135,4.5460,3.9695,0.5660,5,4,2025-09-28,LAA,HOU,2,6,0,1
1,776137,4.5740,4.2920,0.5460,5,4,2025-09-28,SF,COL,4,0,1,1
2,776138,4.0090,3.6745,0.5575,4,4,2025-09-28,ATH,KC,2,9,0,1
3,776139,4.1745,3.8000,0.5535,4,4,2025-09-28,SEA,LAD,1,6,0,1
4,776140,4.6010,4.3770,0.5320,5,4,2025-09-28,NYY,BAL,3,2,1,1
5,776141,4.5785,3.9940,0.5730,5,4,2025-09-28,PHI,MIN,1,1,0,1
6,776142,4.0485,4.0750,0.5050,4,4,2025-09-28,CHC,STL,2,0,1,1
7,776143,4.6575,4.2895,0.5365,5,4,2025-09-28,WSH,CWS,0,8,0,1
8,776144,3.7670,3.8245,0.5120,4,4,2025-09-28,MIL,CIN,4,2,1,1
9,776145,3.8160,3.7355,0.5190,4,4,2025-09-28,TOR,TB,13,4,1,1
